In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [3]:
df = pd.read_csv("../data/processed/telco_clean.csv")

In [4]:
X = df.drop(["customerID", "Churn"], axis=1)

y = df["Churn"].map({
    "No":0,
    "Yes":1
})

In [5]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

In [6]:
numeric_features = X.select_dtypes(
    include=["int64","float64"]
).columns

In [7]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns

In [8]:
numeric_pipeline = Pipeline(

steps=[

("imputer",
 SimpleImputer(strategy="median")),

("scaler",
 StandardScaler())

]

)

In [9]:
categorical_pipeline = Pipeline(

steps=[

("imputer",

SimpleImputer(

strategy="most_frequent"

)),

("encoder",

OneHotEncoder(

handle_unknown="ignore"

))

]

)

In [10]:
preprocessor = ColumnTransformer(

transformers=[

(

"num",

numeric_pipeline,

numeric_features

),

(

"cat",

categorical_pipeline,

categorical_features

)

]

)

In [11]:
models = {

"Logistic Regression":

LogisticRegression(max_iter=1000),

"Decision Tree":

DecisionTreeClassifier(random_state=42),

"Random Forest":

RandomForestClassifier(random_state=42),

"KNN":

KNeighborsClassifier(),

"Naive Bayes":

GaussianNB(),

"SVM":

SVC(probability=True),

"Gradient Boosting":

GradientBoostingClassifier(random_state=42),

"XGBoost":

XGBClassifier(

random_state=42,

eval_metric="logloss"

)

}

In [12]:
results = []

In [13]:
results = []

for name, model in models.items():

    try:

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_test)

        probabilities = pipeline.predict_proba(X_test)[:, 1]

        results.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, predictions),
            "Precision": precision_score(y_test, predictions, zero_division=0),
            "Recall": recall_score(y_test, predictions, zero_division=0),
            "F1 Score": f1_score(y_test, predictions, zero_division=0),
            "ROC AUC": roc_auc_score(y_test, probabilities)
        })

        print(f"✅ {name} completed")

    except Exception as e:

        print(f"❌ {name} failed")
        print(e)

✅ Logistic Regression completed
✅ Decision Tree completed
✅ Random Forest completed
✅ KNN completed
✅ Naive Bayes completed


E:\Data Science\Intelligent-Customer-Churn-Prediction\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


✅ SVM completed
✅ Gradient Boosting completed
✅ XGBoost completed


In [14]:
results_df = pd.DataFrame(results)

In [15]:
results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.803838,0.648485,0.572193,0.607955,0.835929
1,Decision Tree,0.730633,0.493369,0.497326,0.495340,0.656455
2,Random Forest,0.787491,0.630662,0.483957,0.547655,0.813661
3,KNN,0.761194,0.547980,0.580214,0.563636,0.779444
4,Naive Bayes,0.682303,0.447178,0.826203,0.580282,0.804912
5,SVM,0.791756,0.640138,0.494652,0.558069,0.788559
6,Gradient Boosting,0.796731,0.642857,0.529412,0.580645,0.838614
7,XGBoost,0.769012,0.571014,0.526738,0.547983,0.811913


In [16]:
results_df.to_csv(
    "../reports/metrics/model_comparison.csv",
    index=False
)

In [17]:
print(type(results))

print(len(results))

print(results[:2])

<class 'list'>
8
[{'Model': 'Logistic Regression', 'Accuracy': 0.8038379530916845, 'Precision': 0.6484848484848484, 'Recall': 0.5721925133689839, 'F1 Score': 0.6079545454545454, 'ROC AUC': 0.8359290473207676}, {'Model': 'Decision Tree', 'Accuracy': 0.7306325515280739, 'Precision': 0.493368700265252, 'Recall': 0.49732620320855614, 'F1 Score': 0.49533954727030627, 'ROC AUC': 0.6564546438129947}]
